# Phase II, Pair 1: question F — neighbouring points

*Can information from neighbouring points change the forecast, in which form must it be given, and what does the model do with it?*

| | |
|---|---|
| Authors | |
| Status | in progress |
| Phase I result this builds on | Exp 3, 4 and 5 (pressure, a warning flag and neighbours as extra inputs did not improve the drop captured); open question 2 in `phase1-model-behavior/pair-1/experiments/phase2_questions_carlos.md` |

## 1 · Question

Can information from neighbouring points change the forecast, in which form must it be given, and what does the model do with it?

## 2 · Why this question

In Phase I, TiRex-2 forecast the 2 m temperature of each point from its own history only. Giving it more inputs (pressure,
a perfect front flag, neighbouring and upstream points; Exp 3, 4, 5) did not raise the share of the drop it captured
(all about 21 %). We do not know *why*: the model might not see the extra rows at all, it might see them but not use them,
or we might have given them in a form the model cannot use. TiRex-2 accepts neighbours in three different ways (as extra
targets, as past-only covariates, as future-known covariates), and Phase I mostly used past-only channels.

## 3 · Hypothesis and what would count as an answer

**Hypothesis.** Past-only neighbour channels are ignored, as in Phase I. Future covariates and additional targets might
change the forecast: future covariates are read by the reverse (backward-in-time) cell too and reach into the forecast
window, and additional targets are mixed with the point through the variate attention in both directions.

- **If the hypothesis is right, we would see:** for past-only channels a skill gain of about zero, gradients with respect
  to the neighbour channels at the level of a meaningless control channel, and hidden states that barely differ from the
  baseline. For future covariates (shifted, no leakage) and/or extra targets: a skill gain whose confidence interval
  excludes zero, at least for front starts, together with clearly non-zero gradients and hidden-state differences.
- **If it is wrong, we would see:** the same picture for every form: no skill gain, or a gain only in the leakage upper
  bound. If gradients and hidden states move but the skill does not improve, the model *sees* the neighbours but does not
  *use* them for the forecast.

## 4 · Setup

*Imports, paths, model loading.*

### Reading `tirex2.py` first (source checked: `tirex-2==0.3.1`, as pinned in `requirements.txt`, and `main` of `github.com/NX-AI/tirex-2`)

**How the input is grouped.** `PostProcessor.transform_input` packs every `TimeseriesType` of the batch into **one group**:
its target rows first, then its past covariates, then its future covariates. It returns `x` (one row per variate),
`group_vector` (the group id per row) and `target_mask` (True for target rows). `TiRex2._prepare_forward` then sets
`known_covariate_mask = has a value in the last future_len steps & not a target`, so future covariates are "known",
past covariates (NaN in the future part) are not.

**`variate_indices(group_vector, target_mask, known_covariate_mask)`** (in `main`; 0.3.1 computes the same inside the
blocks) returns two index sets, once per forward pass:
- `reverse_idx`: the rows that also go through the **reverse** xLSTM cell of the time mixer, i.e. the known (future)
  covariates (all non-targets if there is no `known_covariate_mask`). Targets and past covariates only run forward in time.
- `attention_idx`: the rows whose group has **more than one row**. Only these rows enter the variate attention;
  a lone univariate series skips it.

**Do several targets exchange information?** Yes. Each `MultivariateBlock` runs (1) the time mixer on every row on its
own, then (2) the `variate_mixer`, an `AttentionBlock` over the rows of each group at every token position. The mask
is `same_group & ~(covariate → target)`: rows of different groups (different samples of the batch) never see each other,
**covariates cannot attend to targets**, but **targets attend to every row of their group, including other targets**.
So rows given as extra targets of the same `TimeseriesType` are mixed with the point in both directions in every block;
they are not just batched. Form (i) can therefore work in principle. Rows given as separate `TimeseriesType`s (as in
`run_forecasts.py`) are only batched and never mix.

Two side effects to keep in mind: each target row is scaled and, if `tta_diff` finds a trend, differenced on its own; and
with `tta_sign_flip` the whole forecast is run a second time on the negated input and averaged.

**How to pass the inputs to `forecast()`.**
`TimeseriesType(target=[n_targets, T], past_covariates=[n_past, T] or None, future_covariates=[n_fut, ≥ T+H] or None)`,
then `model.forecast([ts, ...], prediction_length=H, output_type="numpy")`. Each output is `[n_targets, 9 quantiles, H]`;
row 0 is our point. Future covariates longer than `T+H` are cut.

In [ ]:
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tirex2 import TimeseriesType, load_model

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA = Path(os.environ.get("AIVIS_DATA", ROOT / "data" / "austria_15km"))
CKPT = os.environ.get("TIREX_CKPT", "NX-AI/TiRex-2")  # a local folder works too
DEVICE = os.environ.get("TIREX_DEVICE", "cpu")
RESULTS = ROOT / "data" / "notebooks" / "results" / "06_question_f"
RESULTS.mkdir(parents=True, exist_ok=True)

CONTEXT, HORIZON = 672, 24  # Phase I design: 28 days of history, 24 h forecast
QUANTILES = np.arange(1, 10) / 10
SEED = 0

model = load_model(CKPT, device=DEVICE)  # as in phase1-model-behavior/pair-1/run_forecasts.py
core = model.model  # the TiRex2 backbone
print(
    "input_patch_size",
    core.input_patch_size,
    "| blocks",
    len(core.stack),
    "| context_len",
    core.context_len,
    "| future_len",
    core.future_len,
    "| tta_sign_flip",
    core.tta_sign_flip,
    "| tta_diff",
    core.tta_diff,
)

## 5 · Data

*Which cases (how many dates, how many points), and why this sample is large enough to call a pattern systematic.*

## 6 · Technique 1

*What it shows, and what it cannot show.*

## 7 · Technique 2

*What it shows, and what it cannot show.*

## 8 · Comparing the techniques

*Do they agree? Where do they differ?*

## 9 · Checks

*Does the result hold on other dates, other points, other groups? What did we try to break it with?*

## 10 · Figure decisions

| Figure | Keep / change / drop | Why |
|---|---|---|

## 11 · What it means

*Answer to the question, in plain words. What is still open?*

## 12 · What did not work

*Dead ends and why. They are part of the record.*